# Optical Cavity Design — Interactive Reference

An interactive reference tool for designing optical cavities from first principles,
scaling from tabletop (cm) to LIGO-scale (km).
All physics is implemented in numpy — no optical simulation packages required.

**Target audience:** People with basic knowledge of Gaussian beams, lenses, and mirrors.

**Conventions used throughout:**

| Quantity | Unit |
|----------|------|
| Wavelength | nm |
| Lengths | mm |
| Frequencies / FSR | MHz |
| Linewidths | Hz |
| Power reflectivities R | dimensionless ∈ [0, 1] |
| Amplitude reflectivities r = √R | dimensionless ∈ [0, 1] |

Mirror ROC sign convention: **positive** when the center of curvature faces the incoming beam
(Siegman convention).

---

## Contents

1. [Introduction & motivation](#1.-Introduction-&-motivation)
2. [Cavity basics: finesse, FSR, linewidth, power buildup, coupling](#2.-Cavity-basics)
3. [Stability diagrams](#3.-Cavity-stability)
4. Gaussian beam optics & ABCD formalism *(stub)*
5. [RF sidebands & PDH locking](#5.-RF-sidebands-&-PDH-locking)
6. [Higher-order modes & accidental resonances](#6.-Higher-order-modes-&-accidental-resonances)
7. [Design cost function & decision tree](#7.-Design-cost-function-&-decision-tree)

## 1. Introduction & motivation

Optical cavities are feedback loops made up of mirrors and propagation through space, but basically the same as normal feedback loops.

This notebook builds up cavity design from the simplest two-mirror Fabry-Perot to three- and four-mirror ring geometries. 

Along the way you will see how five coupled quantities: 
finesse, FSR, linewidth, power buildup, and mode structure
constrain every design choice.

**Why build a cavity?**
A cavity multiplies the effective reflectivity of ordinary mirrors: a mirror with
$R = 0.99$ alone transmits 1% per pass, but the same mirror at finesse $\mathcal{F} = 300$
stores each photon for $\sim\mathcal{F}/\pi \approx 100$ round trips and amplifies the
intracavity field by $\sim\mathcal{F}/\pi$ over the input.  This buildup underpins laser gain,
spectroscopic sensitivity, and gravitational-wave signal recycling alike.
The mechanism is simple: each round trip the field interferes with itself;
resonance is the condition where that interference is everywhere constructive.
*(References: Yariv, §4; Siegman, §11.)*

**Running examples used throughout:**

| Example | Geometry | Finesse | λ | Purpose |
|---------|----------|---------|---|---------|
| 2-mirror FP | linear | 10 000 | 1064 nm | reference cavity, critically coupled |
| aLIGO IMC | triangular ring (3-mirror) | ~500 | 1064 nm | input mode cleaner |
| aLIGO OMC | bow-tie ring (4-mirror) | ~400 | 1064 nm | output mode cleaner |

In [ ]:
%matplotlib widget

import sys, os
if '.' not in sys.path:
    sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## 2. Cavity basics

Five quantities fully characterize the linear response of a two-mirror cavity:

| Quantity | Symbol | Formula | Notes |
|----------|--------|---------|-------|
| Free spectral range | FSR | c / 2L | adjacent resonance spacing |
| Finesse | F | π (R₁R₂)^¼ / (1 − (R₁R₂)^½) | determined by mirror losses |
| Linewidth (FWHM) | δν | FSR / F | how narrow the resonance is |
| Peak circulating buildup | B | (1−R₁) / (1−√(R₁R₂))² | circulating / input power |
| Critical coupling | — | r₁ = r₂ (lossless) | zero on-resonance reflection |

The cavity response is described by the **Airy function** — the exact transfer function for
a lossless two-mirror resonator. The intracavity power and reflected power are:

$$P_\text{circ}/P_\text{in} = \frac{1-R_1}{(1-r_1 r_2)^2 + 4 r_1 r_2\sin^2(\pi\Delta\nu/\text{FSR})}$$

$$P_\text{ref}/P_\text{in} = \frac{(r_1-r_2)^2 + 4 r_1 r_2\sin^2(\pi\Delta\nu/\text{FSR})}{(1-r_1 r_2)^2 + 4 r_1 r_2\sin^2(\pi\Delta\nu/\text{FSR})}$$

where $r_i = \sqrt{R_i}$ are amplitude reflectivities and $\Delta\nu$ is the detuning from resonance.

**References:** Yariv, *Optical Electronics in Modern Communications*, §4 ·
Siegman, *Lasers*, §11

---

Adjust the sliders below. The x-axis is in units of the FSR so the resonance structure
remains visible regardless of cavity length.

In [ ]:
from src.cavity import (finesse, fsr, linewidth, power_buildup,
                         coupling_status, circulating_spectrum, reflection_spectrum)

_N    = 6000
_SPAN = 2.5
_x    = np.linspace(-_SPAN, _SPAN, _N)

_fig2, (_ax_c, _ax_r) = plt.subplots(1, 2, figsize=(11, 4))
_fig2.tight_layout(rect=[0, 0, 1, 0.93], pad=2.5)

_sl = dict(style={'description_width': '160px'}, layout=widgets.Layout(width='490px'))
_R1_w = widgets.FloatSlider(value=0.990,  min=0.80, max=0.9999, step=0.001,
                             description='R₁ (input coupler)',
                             readout_format='.4f', **_sl)
_R2_w = widgets.FloatSlider(value=0.9999, min=0.80, max=0.9999, step=0.0001,
                             description='R₂ (end mirror)',
                             readout_format='.4f', **_sl)
_L_w  = widgets.FloatSlider(value=100,    min=10,   max=1000,   step=5,
                             description='L (mm)',
                             readout_format='.0f',  **_sl)
_info2 = widgets.HTML()


def _fmt_hz(hz):
    if hz >= 1e9:  return f'{hz/1e9:.3g} GHz'
    if hz >= 1e6:  return f'{hz/1e6:.3g} MHz'
    if hz >= 1e3:  return f'{hz/1e3:.3g} kHz'
    return f'{hz:.3g} Hz'


def _update2(_change=None):
    R1, R2, L_mm = _R1_w.value, _R2_w.value, _L_w.value
    L    = L_mm * 1e-3
    F    = finesse(R1, R2)
    nu0  = fsr(L)
    lw   = linewidth(F, nu0)
    B    = power_buildup(R1, R2)
    coup = coupling_status(R1, R2)

    delta = _x * nu0
    P_c   = circulating_spectrum(delta, nu0, R1, R2)
    P_r   = reflection_spectrum(delta, nu0, R1, R2)

    _ax_c.clear()
    _ax_r.clear()

    _ax_c.semilogy(_x, P_c / B, color='xkcd:cerulean')
    _ax_c.set_xlabel('Detuning (FSR)')
    _ax_c.set_ylabel('P_circ / P_in  (norm. to peak)')
    _ax_c.set_title('Intracavity power')
    _ax_c.set_xlim(-_SPAN, _SPAN)
    _ax_c.set_ylim(1e-6, 1.18)

    _ax_r.semilogy(_x, P_r, color='xkcd:scarlet')
    _ax_r.set_xlabel('Detuning (FSR)')
    _ax_r.set_ylabel('P_ref / P_in')
    _ax_r.set_title('Reflected power')
    _ax_r.set_xlim(-_SPAN, _SPAN)
    _ax_r.set_ylim(1e-6, 1.05)

    _fig2.canvas.draw_idle()

    coup_html = {
        'critical': '<b style="color:green">critically coupled</b>  (r₁ = r₂)',
        'over':     '<b style="color:royalblue">over-coupled</b>  (r₁ &lt; r₂)',
        'under':    '<b style="color:darkorange">under-coupled</b>  (r₁ &gt; r₂)',
    }[coup]
    _info2.value = (
        '<table style="font-size:14px;border-collapse:collapse;margin-top:8px">'
        f'<tr><td style="padding:3px 24px"><b>Finesse</b></td><td>{F:.0f}</td>'
        f'    <td style="padding:3px 24px"><b>FSR</b></td><td>{_fmt_hz(nu0)}</td></tr>'
        f'<tr><td style="padding:3px 24px"><b>Linewidth</b></td><td>{_fmt_hz(lw)}</td>'
        f'    <td style="padding:3px 24px"><b>Peak buildup</b></td><td>{B:.0f}×</td></tr>'
        f'<tr><td colspan=4 style="padding:3px 24px"><b>Coupling:</b> {coup_html}</td></tr>'
        '</table>'
    )


for _w2 in (_R1_w, _R2_w, _L_w):
    _w2.observe(_update2, names='value')

display(widgets.VBox([_R1_w, _R2_w, _L_w, _info2]))
#display(_fig2.canvas)
_update2()

## 3. Cavity stability

A cavity is stable if a Gaussian beam reproduces itself after each round trip.
The condition depends only on the **g-parameters**:

$$g_i = 1 - \frac{L}{R_i}$$

where $L$ is the cavity length and $R_i$ is the mirror radius of curvature
(positive = concave). The cavity is stable when:

$$0 \le g_1 g_2 \le 1$$

The **round-trip Gouy phase** sets the higher-order mode spacing (Section 6):

$$\psi = \arccos\!\sqrt{g_1 g_2}$$

**Special operating points** lie on the stability boundary ($g_1 g_2 = 0$ or $1$):

| Point | $(g_1, g_2)$ | Practical note |
|-------|-------------|----------------|
| Confocal | $(0, 0)$ | $R = L$ — maximum Gouy phase, best HOM discrimination |
| Planar | $(1, 1)$ | $R \to \infty$ — extremely sensitive to misalignment |
| Hemispherical | $(1, 0)$ or $(0, 1)$ | one flat mirror — common in lasers |
| Concentric | $(-1, -1)$ | $R = L/2$ — huge beam on mirrors |

Practical designs stay well inside the stable region, typically $g_1 g_2 \in [0.1,\, 0.9]$.
Proximity to the stability boundary ($g_1 g_2 = 0$ or $1$) has a concrete physical cost:
beam sizes on the mirrors diverge, alignment sensitivity increases sharply, and maintaining
a locked cavity becomes mechanically demanding.  The **stability margin**
$\min(g_1 g_2,\, 1 - g_1 g_2)$ is the practical figure of merit — the wider the margin,
the more tolerant the cavity is to mirror tilt and length fluctuations.

**LIGO arm cavities** operate in the third quadrant ($g_1 < 0$, $g_2 < 0$), near
$g_1 g_2 \approx 0.85$.  Both mirrors are concave with $R_i$ slightly greater than $L/2$,
so $g_i = 1 - L/R_i < 0$.  The resulting large beam size ($\sim 6$ cm on the test masses)
averages over mirror surface figure errors and reduces the coupling of scattered light
into higher-order modes — the primary physical motivation for this unconventional operating
point.  (LIGO-T010075, LIGO-T0900043.)

**Reference:** Siegman, *Lasers*, Chapters 19–21

---

The left plot shows the operating point on the $g_1$–$g_2$ stability plane (blue = stable region).
The right plot shows the beam caustic — the beam radius profile from M1 to M2.

In [ ]:
from src.stability import (g_params, is_stable, stability_margin,
                            gouy_phase_rt, cavity_mode)
from src.beams import beam_caustic

_WAV = 1064e-9   # metres

# Stability diagram background (computed once)
_gr    = np.linspace(-1.5, 1.5, 400)
_GG1, _GG2 = np.meshgrid(_gr, _gr)
_smask = (_GG1 * _GG2 >= 0) & (_GG1 * _GG2 <= 1)

# Special operating points: (g1, g2, label, text offset)
_SPTS = [
    ( 1.0,  1.0, 'Planar',      ( 5, -14)),
    (-1.0, -1.0, 'Concentric',  (-75,  6)),
    ( 0.0,  0.0, 'Confocal',    ( 6,   6)),
    ( 1.0,  0.0, 'Hemisph.',    ( 6,   6)),
    ( 0.0,  1.0, '',            ( 6,   6)),
]

# Figure
_fig3, (_ax_g, _ax_b) = plt.subplots(1, 2, figsize=(12, 5))
_fig3.tight_layout(rect=[0, 0, 1, 0.93], pad=2.5)

# Controls
_sl3 = dict(style={'description_width': '160px'}, layout=widgets.Layout(width='490px'))
_R1_s = widgets.FloatLogSlider(value=500, base=10,
                                min=np.log10(50), max=6, step=0.025,
                                description='R₁ (mm)',
                                readout_format='.0f', **_sl3)
_R2_s = widgets.FloatLogSlider(value=500, base=10,
                                min=np.log10(50), max=6, step=0.025,
                                description='R₂ (mm)',
                                readout_format='.0f', **_sl3)
_L3_s = widgets.FloatSlider(value=200, min=10, max=2000, step=10,
                             description='L (mm)',
                             readout_format='.0f', **_sl3)
_info3 = widgets.HTML()


def _update3(_change=None):
    R1 = _R1_s.value * 1e-3
    R2 = _R2_s.value * 1e-3
    L  = _L3_s.value * 1e-3
    L_mm = L * 1e3

    mode = cavity_mode(L, R1, R2, wavelength=_WAV)
    g1, g2 = mode['g1'], mode['g2']

    # --- stability diagram ---
    _ax_g.clear()
    _ax_g.contourf(_GG1, _GG2, _smask.astype(float),
                   levels=[0.5, 1.5], colors=['xkcd:light blue'], alpha=0.9, zorder=1)
    _ax_g.contour(_GG1, _GG2, _GG1 * _GG2,
                  levels=[0.0, 1.0], colors=['xkcd:cerulean'], linewidths=1.5, zorder=2)
    _ax_g.axhline(0, color='xkcd:grey', lw=0.5, zorder=0)
    _ax_g.axvline(0, color='xkcd:grey', lw=0.5, zorder=0)

    for (gx, gy, label, ofs) in _SPTS:
        _ax_g.plot(gx, gy, 'o', color='xkcd:dark grey', ms=6, zorder=4)
        if label:
            _ax_g.annotate(label, (gx, gy), textcoords='offset points',
                           xytext=ofs, fontsize=8, color='xkcd:dark grey')

    pt_color = 'xkcd:scarlet' if mode['stable'] else 'xkcd:orange'
    g1p = np.clip(g1, -1.45, 1.45)
    g2p = np.clip(g2, -1.45, 1.45)
    _ax_g.plot(g1p, g2p, 'o', color=pt_color, ms=11, zorder=5)

    _ax_g.set_xlim(-1.5, 1.5)
    _ax_g.set_ylim(-1.5, 1.5)
    _ax_g.set_xlabel('g₁ = 1 − L/R₁')
    _ax_g.set_ylabel('g₂ = 1 − L/R₂')
    _ax_g.set_title('g₁·g₂ stability plane')
    _ax_g.set_aspect('equal')

    # --- beam caustic ---
    _ax_b.clear()

    if mode['stable'] and mode['q1'] is not None:
        z_arr = np.linspace(0, L, 2000)
        w_arr = beam_caustic(mode['q1'], z_arr, _WAV)

        wmax = np.max(w_arr)
        scale, unit = (1e6, 'μm') if wmax < 5e-4 else (1e3, 'mm')

        z_mm = z_arr * 1e3
        yw   = w_arr * scale

        _ax_b.fill_between(z_mm, -yw, yw, alpha=0.3, color='xkcd:cerulean')
        _ax_b.plot(z_mm,  yw, color='xkcd:cerulean', lw=1.5)
        _ax_b.plot(z_mm, -yw, color='xkcd:cerulean', lw=1.5)

        if 0 < mode['d_waist'] < L:
            dw_mm = mode['d_waist'] * 1e3
            w0_str = f'{mode["w0"] * scale:.1f} {unit}'
            _ax_b.axvline(dw_mm, color='xkcd:grey', ls='--', lw=1.2)
            _ax_b.annotate(f'w₀ = {w0_str}', (dw_mm, 0),
                           textcoords='offset points', xytext=(5, 5),
                           fontsize=8, color='xkcd:dark grey')

        _ax_b.axvline(0,    color='k', lw=2.5)
        _ax_b.axvline(L_mm, color='k', lw=2.5)
        _ax_b.set_xlim(-L_mm * 0.06, L_mm * 1.06)
        _ax_b.set_xlabel('Position (mm)')
        _ax_b.set_ylabel(f'Beam radius ({unit})')
        _ax_b.set_title('Beam caustic')
    else:
        _ax_b.text(0.5, 0.5, f'Unstable cavity\ng₁·g₂ = {g1*g2:.3f}',
                   ha='center', va='center', transform=_ax_b.transAxes,
                   fontsize=13, color='xkcd:scarlet')
        _ax_b.set_title('Beam caustic')

    _fig3.canvas.draw_idle()

    # --- readout ---
    if mode['stable'] and mode['w0'] is not None:
        wmax_all = max(mode['w0'], mode['w1'], mode['w2'])
        sc, un = (1e6, 'μm') if wmax_all < 5e-4 else (1e3, 'mm')

        def _fw(v): return f'{v * sc:.1f} {un}'

        gouy   = gouy_phase_rt(g1, g2)
        margin = stability_margin(g1, g2)
        gouy_s   = f'{gouy:.1f}°'   if not np.isnan(gouy)   else '—'
        margin_s = f'{margin:.3f}'        if not np.isnan(margin) else '—'

        _info3.value = (
            '<table style="font-size:13px;border-collapse:collapse;margin-top:6px">'
            f'<tr>'
            f'<td style="padding:3px 16px"><b>g₁</b></td><td>{g1:.4f}</td>'
            f'<td style="padding:3px 16px"><b>g₂</b></td><td>{g2:.4f}</td>'
            f'<td style="padding:3px 16px"><b>g₁·g₂</b></td><td>{g1*g2:.4f}</td>'
            f'</tr><tr>'
            f'<td style="padding:3px 16px"><b>Waist w₀</b></td><td>{_fw(mode["w0"])}</td>'
            f'<td style="padding:3px 16px"><b>w₁ (M1)</b></td><td>{_fw(mode["w1"])}</td>'
            f'<td style="padding:3px 16px"><b>w₂ (M2)</b></td><td>{_fw(mode["w2"])}</td>'
            f'</tr><tr>'
            f'<td style="padding:3px 16px"><b>Waist pos.</b></td>'
            f'<td>{mode["d_waist"]*1e3:.1f} mm from M1</td>'
            f'<td style="padding:3px 16px"><b>Gouy phase</b></td><td>{gouy_s}</td>'
            f'<td style="padding:3px 16px"><b>Stab. margin</b></td><td>{margin_s}</td>'
            f'</tr>'
            '</table>'
        )
    else:
        _info3.value = (
            f'<p style="color:darkorange;font-size:14px">'
            f'<b>Unstable cavity:</b> g₁·g₂ = {g1*g2:.4f} '
            f'(g₁ = {g1:.4f}, g₂ = {g2:.4f})</p>'
        )


for _w3 in (_R1_s, _R2_s, _L3_s):
    _w3.observe(_update3, names='value')

display(widgets.VBox([_R1_s, _R2_s, _L3_s, _info3]))
display(_fig3.canvas)
_update3()

## 4. Gaussian beam optics & ABCD formalism

<!-- TODO -->

This section will cover Gaussian beam propagation using ABCD matrices — inside the cavity
and in the input/output beam paths.  Mode-matching from input beam to cavity eigenmode is
reserved for a future update.

The Gaussian beam tools needed here are already implemented in `src/beams.py`:
complex q-parameter, ABCD propagation, `beam_caustic`.  They are used live in Section 3.

**Reference:** Siegman, *Lasers*, Chapters 17–20

## 5. RF sidebands & PDH locking

Phase modulation at angular frequency $\omega_m$ splits the carrier into a comb of sidebands. The field entering the cavity is

$$E(t) = E_0\,e^{i(\omega t + \Gamma\sin\omega_m t)} = E_0\sum_{n=-\infty}^{\infty} J_n(\Gamma)\,e^{i(\omega + n\omega_m)t},$$

where $\Gamma$ is the modulation depth and $J_n$ are Bessel functions of the first kind. In practice the expansion is truncated at $|n|\leq 3$ (panel D shows the power in each order).

Each frequency component reflects from the cavity with its own complex reflectivity $r_\mathrm{cav}(\delta)$, where $\delta$ is the detuning from the nearest resonance:

$$r_\mathrm{cav}(\delta) = \frac{r_1 - r_2\,e^{i\phi}}{1 - r_1 r_2\,e^{i\phi}}, \qquad \phi = \frac{2\pi\delta}{\mathrm{FSR}}.$$

The reflected power (panel A) is $P_\mathrm{refl} = \sum_n J_n^2\,|r_n|^2$ where $r_n \equiv r_\mathrm{cav}(\delta + n \omega_m/2\pi)$.

**Pound–Drever–Hall demodulation.** The photodetector signal contains a beat at $k\omega_m$ with complex amplitude

$$A_k(\delta) = \sum_n J_n(\Gamma)\,J_{n-k}(\Gamma)\;r_n(\delta)\,r_{n-k}^*(\delta).$$

The PDH error signal (panel B) is $\varepsilon = \mathrm{Re}(A_1\,e^{i\varphi_\mathrm{demod}})$. At $\varphi_\mathrm{demod}=0°$ this gives the standard dispersive discriminant. Rotating by 90° accesses the absorption quadrature.

Panel C shows the in-phase (I, solid) and quadrature (Q, dashed) components at 1f, 2f, and 3f. The 2f signal provides an independent error signal; 3f is useful for diagnostics.

The key requirement for clean PDH operation is that **sidebands must be off cavity resonance**: $\omega_m/2\pi \gg \delta\nu$ (linewidth). With $\omega_m/2\pi \ll \mathrm{FSR}/2$, the sidebands are fully reflected and act as a stable phase reference for the carrier.

*Reference: E. D. Black, Am. J. Phys. **69**, 79 (2001).*

In [ ]:
from src.sidebands import (
    R_from_finesse, reflected_power, demod_amplitudes, pdh_signal,
    sideband_input_spectrum,
)
from src.cavity import fsr as _fsr5

_C5 = 299_792_458.0   # m/s

_sl5 = dict(style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))

_L5_s    = widgets.FloatSlider(value=100, min=10, max=500, step=5,
                               description='L (mm)',
                               readout_format='.0f', **_sl5)
_F5_s    = widgets.FloatLogSlider(value=1000, base=10,
                                  min=1, max=5, step=0.05,
                                  description='Finesse',
                                  readout_format='.0f', **_sl5)
_fmod5_s = widgets.FloatSlider(value=50, min=1, max=300, step=1,
                               description='ωm/2π (MHz)',
                               readout_format='.1f', **_sl5)
_beta5_s = widgets.FloatSlider(value=0.3, min=0.01, max=2.0, step=0.01,
                               description='Γ (mod. depth)',
                               readout_format='.2f', **_sl5)
_phi5_s  = widgets.FloatSlider(value=0, min=-180, max=180, step=1,
                               description='φ_demod (°)',
                               readout_format='.0f', **_sl5)
_info5 = widgets.HTML()

_fig5, _axs5 = plt.subplots(2, 2, figsize=(11, 7))
_fig5.suptitle('Section 5 — RF Sidebands & PDH Locking', fontsize=12)
_fig5.tight_layout(rect=[0, 0, 1, 0.95], pad=3.0)
_axs5 = _axs5.flatten()


def _update5(_=None):
    L_m   = _L5_s.value * 1e-3
    F     = _F5_s.value
    f_mod = _fmod5_s.value * 1e6
    beta  = _beta5_s.value
    phi_d = _phi5_s.value

    fsr_hz = _C5 / (2.0 * L_m)
    lw_hz  = fsr_hz / F
    R      = R_from_finesse(F)

    span  = 8.0 * lw_hz
    delta = np.linspace(-span, span, 4000)

    # dynamic axis units
    if span < 1e6:
        xv, xu = delta / 1e3, 'kHz'
    else:
        xv, xu = delta / 1e6, 'MHz'

    # --- Panel A: reflected power (log scale) ---
    pwr = reflected_power(delta, fsr_hz, R, R, beta, f_mod)
    _axs5[0].cla()
    _axs5[0].semilogy(xv, pwr, color='xkcd:cerulean', lw=1.5)
    _axs5[0].axvline(0, color='k', lw=0.5, ls='--')
    _axs5[0].set_xlabel(f'Detuning ({xu})')
    _axs5[0].set_ylabel('Reflected power')
    _axs5[0].set_title('(A) Reflected power')
    _axs5[0].set_ylim(1e-5, 2)

    # --- Panel B: PDH error signal ---
    err = pdh_signal(delta, fsr_hz, R, R, beta, f_mod,
                     demod_phase_deg=phi_d, harmonic=1)
    _axs5[1].cla()
    _axs5[1].plot(xv, err, color='xkcd:orange', lw=1.5)
    _axs5[1].axhline(0, color='k', lw=0.5)
    _axs5[1].axvline(0, color='k', lw=0.5, ls='--')
    _axs5[1].set_xlabel(f'Detuning ({xu})')
    _axs5[1].set_ylabel('PDH signal (arb)')
    _axs5[1].set_title(f'(B) PDH error signal  [1f · φ = {phi_d:.0f}°]')

    # --- Panel C: I/Q quadratures 1f / 2f / 3f ---
    amps = demod_amplitudes(delta, fsr_hz, R, R, beta, f_mod)
    _axs5[2].cla()
    _c5 = ['xkcd:cerulean', 'xkcd:orange', 'xkcd:green']
    for k, col in zip([1, 2, 3], _c5):
        _axs5[2].plot(xv, amps[f'I{k}'], color=col, ls='-',  lw=1.2, label=f'{k}f I')
        _axs5[2].plot(xv, amps[f'Q{k}'], color=col, ls='--', lw=1.2, label=f'{k}f Q')
    _axs5[2].axhline(0, color='k', lw=0.5)
    _axs5[2].legend(fontsize=7, ncol=3, loc='upper right')
    _axs5[2].set_xlabel(f'Detuning ({xu})')
    _axs5[2].set_ylabel('Amplitude (arb)')
    _axs5[2].set_title('(C) Quadratures  1f / 2f / 3f')

    # --- Panel D: input sideband spectrum ---
    freqs, powers = sideband_input_spectrum(f_mod, beta, max_order=3)
    bar_w = f_mod * 0.22 / 1e6   # MHz
    _axs5[3].cla()
    _axs5[3].bar(freqs / 1e6, powers, width=bar_w, color='xkcd:purple')
    _axs5[3].set_xlabel('Frequency offset from carrier (MHz)')
    _axs5[3].set_ylabel('Power  |J_n(Γ)|²')
    _axs5[3].set_title('(D) Input field spectrum')
    _axs5[3].set_xlim(-3.8 * f_mod / 1e6, 3.8 * f_mod / 1e6)

    # --- info bar ---
    ratio = f_mod / lw_hz
    ratio_col = 'green' if ratio > 10 else 'darkorange'
    _info5.value = (
        '<table style="font-size:13px;border-collapse:collapse;margin-top:6px">'
        '<tr>'
        f'<td style="padding:3px 20px"><b>FSR</b></td>'
        f'<td>{fsr_hz/1e9:.3f} GHz</td>'
        f'<td style="padding:3px 20px"><b>Linewidth</b></td>'
        f'<td>{lw_hz/1e3:.2f} kHz</td>'
        f'<td style="padding:3px 20px"><b>ωm/2π / lw</b></td>'
        f'<td style="color:{ratio_col}"><b>{ratio:.0f}</b></td>'
        '</tr>'
        '</table>'
    )

    _fig5.canvas.draw_idle()


for _w5 in (_L5_s, _F5_s, _fmod5_s, _beta5_s, _phi5_s):
    _w5.observe(_update5, names='value')

display(widgets.VBox([
    widgets.HBox([_L5_s, _F5_s]),
    widgets.HBox([_fmod5_s, _beta5_s, _phi5_s]),
    _info5,
]))
#display(_fig5.canvas)
_update5()

## 6. Higher-order modes & accidental resonances

The Gouy phase $\psi$ accumulated per round trip (Section 3) shifts every transverse mode
group away from the nearest carrier resonance.  In a 2-mirror linear cavity, all HG modes
of the same order $q = m + n$ are degenerate and resonate at:

$$\Delta f_q = \frac{q\,\psi}{\pi}\,\text{FSR} \pmod{\text{FSR}}, \qquad
\psi = \arccos\!\sqrt{g_1 g_2}$$

(Siegman, *Lasers*, eq. 21.42)

**Color convention:** blue = TEM$_{00}$, green = HOMs, orange = RF sidebands.

An **accidental resonance** occurs when an RF sideband at $\pm n\,f_\text{mod}$ from a
carrier overlaps a HOM resonance.  The sideband power then leaks into a spatial mode that
cannot be detected on a single-mode photodetector, degrading the PDH error signal.
The tool below detects and flags these collisions in real time.

**Reference:** Siegman, *Lasers*, §21 · LIGO T-docs

---

Adjust ROC and cavity length to shift the HOM comb relative to fixed sidebands.
The display spans 2 FSRs; carrier resonances appear at 0, FSR, and 2·FSR.

In [ ]:
from src.hom import tem00_spectrum, hom_spectrum
from src.stability import g_params, gouy_phase_rt, stability_margin
from src.cavity import fsr as _fsr6, linewidth as _lw6

_fig6, _ax6 = plt.subplots(1, 1, figsize=(12, 4))
_fig6.tight_layout(rect=[0, 0, 1, 0.93])

_sl6 = dict(style={'description_width': '170px'}, layout=widgets.Layout(width='500px'))
_R6   = widgets.FloatLogSlider(value=500, base=10,
                                min=np.log10(50), max=4, step=0.025,
                                description='ROC (mm, symmetric)',
                                readout_format='.0f', **_sl6)
_L6   = widgets.FloatSlider(value=200, min=50, max=1000, step=10,
                              description='L (mm)',
                              readout_format='.0f', **_sl6)
_F6   = widgets.FloatLogSlider(value=1000, base=10, min=2, max=4.7, step=0.05,
                                description='Finesse',
                                readout_format='.0f', **_sl6)
_ORD6 = widgets.IntSlider(value=5, min=1, max=10, step=1,
                           description='Max mode order', **_sl6)
_FM6  = widgets.FloatSlider(value=30, min=1, max=500, step=1,
                              description='ωm/2π (MHz)',
                              readout_format='.0f', **_sl6)
_info6 = widgets.HTML()


def _fmthz6(hz):
    if hz >= 1e9: return f'{hz/1e9:.3g} GHz'
    if hz >= 1e6: return f'{hz/1e6:.3g} MHz'
    if hz >= 1e3: return f'{hz/1e3:.3g} kHz'
    return f'{hz:.3g} Hz'


def _update6(_change=None):
    R    = _R6.value  * 1e-3
    L    = _L6.value  * 1e-3
    F    = float(_F6.value)
    mord = _ORD6.value
    fmod = _FM6.value * 1e6

    g1, g2   = g_params(L, R, R)
    nu0      = _fsr6(L)
    lw       = _lw6(F, nu0)
    gouy     = gouy_phase_rt(g1, g2)
    margin   = stability_margin(g1, g2)
    nu0_mhz  = nu0  / 1e6
    fmod_mhz = fmod / 1e6

    _ax6.clear()

    if not (0.0 < g1 * g2 < 1.0):
        _ax6.text(0.5, 0.5,
                  f'Unstable cavity\ng₁·g₂ = {g1*g2:.3f}',
                  ha='center', va='center', transform=_ax6.transAxes,
                  fontsize=13, color='xkcd:scarlet')
        _ax6.set_title('Cavity transfer function')
        _fig6.canvas.draw_idle()
        _info6.value = (
            f'<p style="color:darkorange;font-size:14px">'
            f'<b>Unstable:</b> g₁·g₂ = {g1*g2:.4f}</p>'
        )
        return

    # Adaptive resolution: ~20 samples per linewidth, capped at 40 000
    N    = min(40000, max(8000, int(20 * nu0 / lw)))
    freq = np.linspace(0, 2.0 * nu0, N)
    freq_mhz = freq / 1e6

    tem_s            = tem00_spectrum(freq, nu0, F)
    hom_s, hom_peaks = hom_spectrum(freq, nu0, F, g1, g2, mord)

    _ax6.fill_between(freq_mhz, tem_s, alpha=0.45, color='xkcd:cerulean')
    _ax6.fill_between(freq_mhz, hom_s, alpha=0.35, color='xkcd:green')
    _ax6.semilogy(freq_mhz, tem_s, color='xkcd:cerulean', lw=1.0, label='TEM₀₀')
    _ax6.semilogy(freq_mhz, hom_s, color='xkcd:green',   lw=1.0, label='HOMs')

    # RF sideband lines: ±1f, ±2f, ±3f from carriers at 0, FSR, 2·FSR
    for carrier_k in [0, 1, 2]:
        fc = carrier_k * nu0_mhz
        for h in [1, 2, 3]:
            for sign in [+1, -1]:
                fs = fc + sign * h * fmod_mhz
                if 0 <= fs <= 2 * nu0_mhz:
                    _ax6.axvline(fs, color='xkcd:orange', ls='--',
                                 lw=1.3, alpha=0.9 / h)

    _ax6.axvline(-9999, color='xkcd:orange', ls='--', lw=1.3, label='RF sidebands')

    # Annotate first in-range peak for each HOM order
    labeled = set()
    for (f0, q, _) in sorted(hom_peaks, key=lambda x: x[0]):
        f0_mhz = f0 / 1e6
        if 0 < f0_mhz < 2 * nu0_mhz and q not in labeled:
            _ax6.annotate(
                f'q={q}', (f0_mhz, 1.04),
                fontsize=7.5, color='xkcd:dark green',
                ha='center', va='bottom', rotation=90
            )
            labeled.add(q)

    _ax6.set_xlim(0, 2 * nu0_mhz)
    _ax6.set_ylim(1e-6, 1.55)
    _ax6.set_xlabel('Frequency (MHz)')
    _ax6.set_ylabel('Normalised power')
    _ax6.set_title('Cavity transfer function — 2 FSR span')
    _ax6.legend(loc='upper right', fontsize=9)
    _fig6.canvas.draw_idle()

    # Accidental resonance detection: sideband within 3 linewidths of any HOM
    thr = 3.0 * lw
    accidents = []
    seen_acc = set()
    for carrier_k in [0, 1, 2]:
        fc = carrier_k * nu0
        for h in [1, 2, 3]:
            for sign in [+1, -1]:
                fs = fc + sign * h * fmod
                for (f0, q, _) in hom_peaks:
                    if abs(fs - f0) < thr:
                        key = (round(fs / 1e6, 1), q)
                        if key not in seen_acc:
                            seen_acc.add(key)
                            accidents.append(
                                f'⚠ {h}f sideband @ {fs/1e6:.1f} MHz '
                                f'≈ HOM q={q} @ {f0/1e6:.1f} MHz'
                            )

    gouy_s   = f'{gouy:.1f}°'  if not np.isnan(gouy)   else '—'
    margin_s = f'{margin:.3f}'       if not np.isnan(margin) else '—'
    acc_html = ''
    if accidents:
        acc_html = (
            '<tr><td colspan=8 style="padding:2px 16px;'
            'color:darkorange;font-size:13px">'
            + '<br>'.join(accidents[:4]) + '</td></tr>'
        )

    _info6.value = (
        '<table style="font-size:13px;border-collapse:collapse;margin-top:6px">'
        '<tr>'
        f'<td style="padding:3px 16px"><b>FSR</b></td><td>{_fmthz6(nu0)}</td>'
        f'<td style="padding:3px 16px"><b>Linewidth</b></td><td>{_fmthz6(lw)}</td>'
        f'<td style="padding:3px 16px"><b>Gouy ψ</b></td><td>{gouy_s}</td>'
        f'<td style="padding:3px 16px"><b>Stab. margin</b></td><td>{margin_s}</td>'
        '</tr>' + acc_html + '</table>'
    )


for _w6 in (_R6, _L6, _F6, _ORD6, _FM6):
    _w6.observe(_update6, names='value')

display(widgets.VBox([_R6, _L6, _F6, _ORD6, _FM6, _info6]))
#display(_fig6.canvas)
_update6()

## 7. Design cost function & decision tree

Every cavity design involves simultaneous trade-offs between competing objectives. This section collects the physics from Sections 2–6 into a single multi-objective score and lets you explore how adjusting the design parameters changes each objective.

### Five design objectives

| Objective | What it measures | Score = 1 when… |
|-----------|-----------------|-----------------|
| **Stability** | Distance from stability boundary | $g_1 g_2 = 0.5$ (centre of stable region) |
| **HOM avoidance** | Min. separation between any RF sideband and any HOM peak, in linewidths | All separations $> 10\,\delta\nu$ |
| **PDH slope** | Sideband placement: $f_\text{mod} / \delta\nu$ | $f_\text{mod} \geq 10\,\delta\nu$ |
| **Power buildup** | Log-normalised buildup $B$ | $B \geq 10^4$ |
| **Footprint** | Cavity length $L$ | $L \leq 100\,\text{mm}$ |

### Conflicts and trade-offs

- **Finesse vs. footprint:** High finesse (low $\delta\nu$) makes PDH easier and buildup larger, but requires precise mirror specs and longer cavities to avoid being near stability boundaries.
- **Stability vs. HOM avoidance:** Moving away from the stability boundary (towards $g_1 g_2 = 0.5$) increases the Gouy phase shift, which spreads HOMs away from each other — generally *improving* HOM avoidance. These two objectives are *aligned*.
- **Finesse vs. HOM avoidance:** Higher finesse (narrower linewidth) makes accidental resonances *more* dangerous: the same sideband-HOM overlap that was harmless at $F=100$ becomes catastrophic at $F=10\,000$.
- **Footprint vs. PDH slope:** Shorter cavities have larger FSR and therefore wider linewidths. For a fixed $f_\text{mod}$, shorter $L$ makes the PDH condition $f_\text{mod} \gg \delta\nu$ *harder* to satisfy.

### Starter decision tree

```
Cavity design
├─ 1. Cavity type
│   ├─ 2-mirror linear → reference cavities, filters, high-finesse FP
│   ├─ 3-mirror ring (triangular) → mode cleaners, e.g. aLIGO IMC
│   └─ 4-mirror ring (bow-tie) → output mode cleaners, e.g. aLIGO OMC
├─ 2. Scale (fixes FSR and linewidth at given finesse)
│   ├─ Tabletop < 1 m → large FSR; sideband condition requires high f_mod
│   └─ Room-scale > 1 m → narrow linewidth; HOM avoidance becomes tighter
├─ 3. Finesse (mirror specs)
│   ├─ Low (F < 500) → large linewidth; tolerant to HOM overlaps
│   └─ High (F > 5 000) → narrow linewidth; PDH & HOM avoidance are demanding
├─ 4. Mirror ROCs (geometry)
│   ├─ Equal R, symmetric → waist centred; beam equal on both mirrors
│   └─ Asymmetric → waist shifted; allows control of beam size at each mirror
└─ 5. Modulation frequency
    ├─ Must satisfy f_mod ≫ δν (sideband condition)
    └─ Must avoid landing on HOM resonances at any harmonic 1f, 2f, 3f
```

Adjust the sliders below to see how each design choice changes the five objective scores.

### Design algorithm — five-step heuristic

The greedy optimizer below formalises the following recipe. Each step corresponds to one
physical constraint; the ordering reflects which decisions are hardest to undo.

1. **Stability first.** Choose mirror ROCs and cavity length so that
   $g_1 g_2 \in [0.3, 0.7]$.  Stability is a hard constraint — everything else
   is secondary.  A stability margin below $0.1$ makes the cavity practically
   unlockable.

2. **Set the finesse** (mirror specifications).  High finesse gives a narrow
   linewidth and high power buildup, but it makes accidental HOM resonances more
   damaging (a fixed sideband–HOM frequency gap is now measured in more
   linewidths).  Choose finesse before optimizing the modulation frequency.

3. **Choose the modulation frequency.**  The PDH sideband condition requires
   $f_\mathrm{mod} \gg \delta\nu$; at least $10\times$ the linewidth is a safe
   working rule.  Given finesse and cavity length, this sets the *minimum*
   $f_\mathrm{mod}$.

4. **Check HOM avoidance.**  With $f_\mathrm{mod}$ set, scan the Gouy phase
   (Section 6 widget) and confirm that no harmonic $1f, 2f, 3f$ of the
   modulation frequency falls within a few linewidths of any HOM peak up to
   order $q = 5$.  If a collision exists, adjust ROC or $f_\mathrm{mod}$.

5. **Iterate.**  The optimizer below runs this loop automatically: each accepted
   move corresponds to one of the steps above, and the log tells you which
   objective improved.  Use the weight sliders in the widget above to tell the
   algorithm which objectives matter most for your application.

In [ ]:
from src.design import design_scores, OBJECTIVES
from src.stability import g_params
from src.cavity import fsr as _fsr7, linewidth as _lw7

# ── Preset designs ──────────────────────────────────────────────────────────
_PRESETS7 = {
    'Custom': None,
    '2-mirror FP (F=10k, L=100mm)':    dict(L_mm=100,  R1_mm=500,  R2_mm=500,  F=10000, f_mod_mhz=50),
    '2-mirror FP (F=1k, L=100mm)':     dict(L_mm=100,  R1_mm=500,  R2_mm=500,  F=1000,  f_mod_mhz=50),
    'IMC-like (F=500, L=250mm)':        dict(L_mm=250,  R1_mm=800,  R2_mm=800,  F=500,   f_mod_mhz=25),
    'OMC-like (F=400, L=500mm)':        dict(L_mm=500,  R1_mm=2000, R2_mm=2000, F=400,   f_mod_mhz=40),
    'Near-concentric (risky)':          dict(L_mm=200,  R1_mm=102,  R2_mm=102,  F=1000,  f_mod_mhz=30),
}

# ── Controls ─────────────────────────────────────────────────────────────────
_sl7p = dict(style={'description_width': '140px'}, layout=widgets.Layout(width='420px'))
_sl7w = dict(style={'description_width': '140px'}, layout=widgets.Layout(width='360px'))

_L7    = widgets.FloatSlider(value=100, min=10, max=500, step=5,
                             description='L (mm)', readout_format='.0f', **_sl7p)
_R1_7  = widgets.FloatLogSlider(value=500, base=10, min=np.log10(50), max=5, step=0.025,
                                description='R₁ (mm)', readout_format='.0f', **_sl7p)
_R2_7  = widgets.FloatLogSlider(value=500, base=10, min=np.log10(50), max=5, step=0.025,
                                description='R₂ (mm)', readout_format='.0f', **_sl7p)
_F7    = widgets.FloatLogSlider(value=1000, base=10, min=1, max=5, step=0.05,
                                description='Finesse', readout_format='.0f', **_sl7p)
_fm7   = widgets.FloatSlider(value=50, min=1, max=300, step=1,
                             description='ωm/2π (MHz)', readout_format='.1f', **_sl7p)

_WS = {}
for _k, _lbl in OBJECTIVES:
    _WS[_k] = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1,
                                  description=f'w: {_lbl}', readout_format='.1f', **_sl7w)

_preset7 = widgets.Dropdown(options=list(_PRESETS7.keys()), value='Custom',
                             description='Preset:',
                             style={'description_width': '60px'},
                             layout=widgets.Layout(width='360px'))
_info7   = widgets.HTML()

# ── Figure ───────────────────────────────────────────────────────────────────
_fig7, (_ax7r, _ax7b) = plt.subplots(1, 2, figsize=(11, 5),
                                     subplot_kw=dict(polar=False))
_ax7r.remove()
_ax7r = _fig7.add_subplot(121, polar=True)
_fig7.tight_layout(rect=[0, 0, 1, 0.95], pad=3.0)


def _update7(_=None):
    L_mm  = _L7.value
    R1_mm = _R1_7.value
    R2_mm = _R2_7.value
    F     = _F7.value
    fmod  = _fm7.value
    weights = {k: _WS[k].value for k, _ in OBJECTIVES}

    s = design_scores(L_mm, R1_mm, R2_mm, F, fmod, weights=weights)
    keys   = [k for k, _ in OBJECTIVES]
    labels = [lbl for _, lbl in OBJECTIVES]
    vals   = [s[k] for k in keys]

    # ── Radar chart ─────────────────────────────────────────────────────────
    N      = len(OBJECTIVES)
    angles = [n / N * 2 * np.pi for n in range(N)] + [0]
    vplot  = vals + [vals[0]]

    _ax7r.cla()
    _ax7r.set_theta_offset(np.pi / 2)
    _ax7r.set_theta_direction(-1)
    _ax7r.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=9)
    _ax7r.set_ylim(0, 1)
    _ax7r.set_yticks([0.25, 0.5, 0.75, 1.0])
    _ax7r.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=7)
    _ax7r.plot(angles, vplot, 'o-', color='xkcd:cerulean', lw=2)
    _ax7r.fill(angles, vplot, alpha=0.25, color='xkcd:cerulean')
    _ax7r.plot(angles, [1.0] * (N + 1), '--', color='xkcd:light grey', lw=1)
    _ax7r.set_title(f'Total: {s["total"]:.2f}', pad=18, fontsize=11)

    # ── Bar chart ────────────────────────────────────────────────────────────
    _ax7b.cla()
    colors = ['xkcd:cerulean' if v > 0.7 else ('xkcd:orange' if v > 0.4 else 'xkcd:scarlet')
              for v in vals]
    _ax7b.bar(range(N), vals, color=colors, width=0.6)
    _ax7b.set_xticks(range(N))
    _ax7b.set_xticklabels(labels, fontsize=9, rotation=15, ha='right')
    _ax7b.set_ylim(0, 1.1)
    _ax7b.axhline(1.0, color='xkcd:light grey', lw=1, ls='--')
    _ax7b.set_ylabel('Score')
    _ax7b.set_title('Individual scores', fontsize=10)
    for i, v in enumerate(vals):
        _ax7b.text(i, v + 0.03, f'{v:.2f}', ha='center', va='bottom', fontsize=9)

    _fig7.canvas.draw_idle()

    # ── Info bar ─────────────────────────────────────────────────────────────
    g1, g2   = g_params(L_mm*1e-3, R1_mm*1e-3, R2_mm*1e-3)
    fsr_hz   = _fsr7(L_mm*1e-3)
    lw_hz    = _lw7(F, fsr_hz)
    stable   = 0 <= g1*g2 <= 1

    stable_s = ('<b style="color:green">stable</b>' if stable
                else '<b style="color:red">UNSTABLE</b>')
    _info7.value = (
        '<table style="font-size:13px;border-collapse:collapse;margin-top:6px">'
        '<tr>'
        f'<td style="padding:3px 18px"><b>g₁·g₂</b></td><td>{g1*g2:.4f}  {stable_s}</td>'
        f'<td style="padding:3px 18px"><b>FSR</b></td><td>{fsr_hz/1e9:.3f} GHz</td>'
        f'<td style="padding:3px 18px"><b>Linewidth</b></td><td>{lw_hz/1e3:.1f} kHz</td>'
        f'<td style="padding:3px 18px"><b>ωm/2π / lw</b></td><td>{fmod*1e6/lw_hz:.0f}</td>'
        '</tr>'
        '</table>'
    )


def _on_preset7(change):
    val = change['new']
    if val == 'Custom' or _PRESETS7[val] is None:
        return
    p = _PRESETS7[val]
    for w, key in [(_L7,'L_mm'), (_R1_7,'R1_mm'), (_R2_7,'R2_mm'),
                   (_F7,'F'), (_fm7,'f_mod_mhz')]:
        w.value = p[key]


_preset7.observe(_on_preset7, names='value')

for _w7 in [_L7, _R1_7, _R2_7, _F7, _fm7] + list(_WS.values()):
    _w7.observe(_update7, names='value')

_param_col  = widgets.VBox([_L7, _R1_7, _R2_7, _F7, _fm7])
_weight_col = widgets.VBox([_WS[k] for k, _ in OBJECTIVES])
display(widgets.VBox([
    _preset7,
    widgets.HBox([_param_col, _weight_col]),
    _info7,
]))
#display(_fig7.canvas)
_update7()

In [ ]:
# ── Greedy design optimizer ──────────────────────────────────────────────────
# Uses the current widget slider values as the starting point.
# Run the widget above first, then execute this cell.
from src.design import optimize_design, design_scores, OBJECTIVES

_start   = dict(L_mm=_L7.value, R1_mm=_R1_7.value, R2_mm=_R2_7.value,
                F=_F7.value, f_mod_mhz=_fm7.value)
_weights = {k: _WS[k].value for k, _ in OBJECTIVES}

_opt = optimize_design(**_start, weights=_weights)

print('=== Greedy optimizer log ===')
for line in _opt['history']:
    print(line)

r = _opt['result']
print()
print(f"{'Parameter':<14}  {'Start':>12}  {'Optimized':>12}")
print(f"{'-'*42}")
print(f"{'L (mm)':<14}  {_start['L_mm']:>12.1f}  {r['L_mm']:>12.1f}")
print(f"{'R1 (mm)':<14}  {_start['R1_mm']:>12.1f}  {r['R1_mm']:>12.1f}")
print(f"{'R2 (mm)':<14}  {_start['R2_mm']:>12.1f}  {r['R2_mm']:>12.1f}")
print(f"{'Finesse':<14}  {_start['F']:>12.0f}  {r['F']:>12.0f}")
print(f"{'f_mod (MHz)':<14}  {_start['f_mod_mhz']:>12.1f}  {r['f_mod_mhz']:>12.1f}")
print()
_s0 = design_scores(**_start)
print(f"{'Objective':<20}  {'Start':>8}  {'Optimized':>10}")
print(f"{'-'*42}")
for _k, _lbl in OBJECTIVES:
    print(f"{_lbl:<20}  {_s0[_k]:>8.3f}  {r['scores'][_k]:>10.3f}")
print(f"{'-'*42}")
print(f"{'Total':<20}  {_s0['total']:>8.3f}  {r['total']:>10.3f}")